In [ ]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re
import json
import time


def scrape_laws(start_page=1, max_pages=5, output_file="laws.json"):
    all_data = []
    current_auto_id = 1

    # Chrome options
    options = uc.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--start-maximized")

    driver = uc.Chrome(options=options)
    driver.set_page_load_timeout(60)

    # Load dữ liệu cũ nếu có
    try:
        with open(output_file, "r", encoding="utf-8") as f:
            existing_data = json.load(f)
            all_data.extend(existing_data)
            if existing_data:
                current_auto_id = max(item.get("auto_id", 0) for item in existing_data) + 1
    except FileNotFoundError:
        pass

    # Mở trang đầu tiên
    url = "https://thuvienphapluat.vn/page/tim-van-ban.aspx?keyword=&area=0&match=True&type=0&status=0&signer=0&sort=1&lan=1&scan=0&org=0&fields=0"
    driver.get(url)

    for page_num in range(start_page, max_pages + 1):
        print(f"Đang xử lý trang {page_num}...")

        # Chờ danh sách hiện
        WebDriverWait(driver, 30).until(
            EC.presence_of_all_elements_located((By.XPATH, '//a[@onclick="Doc_CT(MemberGA)"]'))
        )

        # Lặp qua từng link
        idx = 0
        while True:
            try:
                law_links = driver.find_elements(By.XPATH, '//a[@onclick="Doc_CT(MemberGA)"]')
                if idx >= len(law_links):
                    break  # hết link trên trang này

                link = law_links[idx]
                href = link.get_attribute("href")
                idx += 1

                if not href:
                    continue

                # Mở chi tiết văn bản
                retry = 0
                while retry < 3:
                    try:
                        driver.get(href)
                        WebDriverWait(driver, 30).until(
                            EC.presence_of_element_located((By.XPATH, '//div[@class="content1"]'))
                        )
                        break
                    except Exception:
                        retry += 1
                        print(f"Retry load {href} lần {retry}")
                        time.sleep(2)

                # Nếu sau 3 lần vẫn fail thì bỏ qua
                if retry == 3:
                    print(f"Bỏ qua link: {href}")
                    driver.back()
                    continue

                # Parse nội dung
                data_item = {}
                container = driver.find_element(By.XPATH, '//div[@class="content1"]')
                all_elements = container.find_elements(By.XPATH, ".//p | .//table") 
                content_list = []

                for el in all_elements:
                    text = el.text.strip()
                    if not text:
                        continue
                    match = re.search(r"Số:\s*(.+)", text)
                    if match:
                        data_item["law_id"] = match.group(1).strip()
                    else:
                        content_list.append(text)

                data_item["content"] = "\n".join(content_list)  # mỗi p/table 1 dòng
                data_item["auto_id"] = current_auto_id
                current_auto_id += 1

                all_data.append(data_item)

                # Lưu sau mỗi mục
                with open(output_file, "w", encoding="utf-8") as f:
                    json.dump(all_data, f, ensure_ascii=False, indent=4)

                print(f"Đã xử lý xong văn bản {data_item.get('law_id', 'N/A')} (auto_id={data_item['auto_id']})")

                # Quay lại danh sách
                driver.back()
                WebDriverWait(driver, 30).until(
                    EC.presence_of_all_elements_located((By.XPATH, '//a[@onclick="Doc_CT(MemberGA)"]'))
                )

            except Exception as e:
                print(f"Lỗi tại mục {idx} trên trang {page_num}: {e}")
                try:
                    driver.back()
                except:
                    pass
                idx += 1

        # Sang trang tiếp theo
        try:
            next_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, '//a[text()="Trang sau"]'))
            )
            next_button.click()
        except:
            print("Không còn trang tiếp theo.")
            break

    print("Hoàn tất cào dữ liệu!")
    driver.quit()



In [ ]:
scrape_laws(start_page=1, max_pages=, output_file="laws_full.json")

Đang xử lý trang 1...
Đã xử lý xong văn bản N/A (auto_id=1)
Đã xử lý xong văn bản 7149/CĐ-BCT (auto_id=2)
Đã xử lý xong văn bản 1329/QĐ-TTPVHCC (auto_id=3)
Đã xử lý xong văn bản 505/TB-VPCP (auto_id=4)
Đã xử lý xong văn bản 503/TB-VPCP (auto_id=5)
Đã xử lý xong văn bản 170/CĐ-TTg (auto_id=6)
Đã xử lý xong văn bản 66.4/2025/NQ-CP (auto_id=7)
Đã xử lý xong văn bản 130/KH-BCĐTKNQ18 (auto_id=8)
Đã xử lý xong văn bản 2109/QĐ-TTg (auto_id=9)
Đã xử lý xong văn bản 169/CĐ-TTg (auto_id=10)
Đã xử lý xong văn bản 168/CĐ-TTg (auto_id=11)
Đã xử lý xong văn bản N/A (auto_id=12)
Đã xử lý xong văn bản 2108/QĐ-TTg (auto_id=13)
Đã xử lý xong văn bản N/A (auto_id=14)
Đã xử lý xong văn bản 249/2025/NĐ-CP (auto_id=15)
Đã xử lý xong văn bản 117/2025/QĐ-UBND (auto_id=16)
Đã xử lý xong văn bản 290/NQ-CP (auto_id=17)
Đã xử lý xong văn bản 25150/TB-CHQ (auto_id=18)
Đã xử lý xong văn bản 496/TB-VPCP (auto_id=19)
Đã xử lý xong văn bản 118/2025/QĐ-UBND (auto_id=20)
Đang xử lý trang 2...
Đã xử lý xong văn bản 28/CT